In [ ]:

from pyspark.sql import functions as F

df_organizations = spark.read.csv(
    "Files/organizations.csv", header=True
)

display(df_organizations)



In [ ]:
df_organizations_transformed = (
    df_organizations

    # Clean hospital name
    .withColumn("NAME", F.trim("NAME"))

    # Clean city name
    .withColumn("CITY", F.trim("CITY"))

    # Standardize state
    .withColumn(
        "STATE",
        F.upper(F.trim("STATE"))
    )

    # Create combined location
    .withColumn(
        "FULL_LOCATION",
        F.concat_ws(
            ", ",
            F.col("CITY"),
            F.col("STATE")
        )
    )

    # Convert geographical coordinates
    .withColumn(
        "LAT",
        F.col("LAT").cast("double")
    )
    .withColumn(
        "LON",
        F.col("LON").cast("double")
    )

    # Preserve postal codes as text
    .withColumn(
        "ZIP",
        F.col("ZIP").cast("string")
    )
)

In [ ]:

display(df_organizations_transformed)

In [ ]:

path = spark.conf.get('spark.executorEnv.HC_Goldpath')
display(path)


In [ ]:

(
    df_organizations_transformed.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "Healthcare_DE_Project.HC_Gold.dbo.DIM_Organizations"
    )
)
